# 01 — Bronze Ingest

## Story beat: "Land the nightly POS export"

Contoso's stores close at midnight. The POS system drops three CSV files into a folder — **stores**, **products**, and **sales**. Our job in the **bronze** layer is simple: **get the data into OneLake as Delta tables with zero transformation**. We preserve every column, every typo, every odd date format. Bronze is the audit trail.

---

## Why bronze matters

- **Reproducibility** — if silver logic changes, you can rebuild from bronze.
- **Lineage** — Purview and Fabric lineage show where raw data entered the estate.
- **Schema-on-read** — CSVs land as files first (`Files/bronze/`); this notebook promotes them to **managed Delta tables** under `Tables/bronze/`.

---

## Prerequisites

- `00_setup` completed (schemas exist, CSVs in `Files/bronze/`).
- Default lakehouse **`lh_retail`** attached.

> **Presenter note:** *"Notice we never copy data out of OneLake. The CSV bytes were uploaded to Files; Spark reads them and writes Delta parquet beside them — still OneLake, still one tenant storage account."*

In [ ]:
%run 00_setup

## Step 1 — Read CSVs and write Delta tables

For each source file we:
1. **Read** with header + inferred schema (quick start; production would use explicit schemas).
2. **Stamp** `_ingested_at` — metadata column proving when the row landed.
3. **Write** as Delta with `mode("overwrite")` — idempotent for demo re-runs.

| Table | Source file | What it contains |
| --- | --- | --- |
| `bronze.stores` | `stores.csv` | Store id, name, city, region |
| `bronze.products` | `products.csv` | Product id, name, category, unit price |
| `bronze.sales` | `sales.csv` | Sale lines: qty, discount, timestamp |

> **Watch the lakehouse Explorer:** After this cell, three new tables appear under **Tables → bronze**. No separate "import to warehouse" step.

In [ ]:
from pyspark.sql import functions as F

raw = {
    "stores":   f"{BRONZE_FILES}/stores.csv",
    "products": f"{BRONZE_FILES}/products.csv",
    "sales":    f"{BRONZE_FILES}/sales.csv",
}
for name, path in raw.items():
    df = (
        spark.read.option("header", True).option("inferSchema", True).csv(path)
        .withColumn("_ingested_at", F.current_timestamp())
    )
    df.write.mode("overwrite").format("delta").saveAsTable(f"{BRONZE_SCHEMA}.{name}")
    print(f"{BRONZE_SCHEMA}.{name}: {df.count()} rows")

## Step 2 — Verify bronze is queryable

The same Delta files are now visible to:
- **Spark** (this notebook)
- **Lakehouse SQL analytics endpoint** (read-only T-SQL — Module 2)
- **Power BI Direct Lake** (later — Module 4)

That is **zero-copy**: one set of parquet files, many engines.

---

## ✅ Success looks like

- Row counts printed for all three tables.
- Sample sales rows display below.

**Next notebook:** `02_silver_transform` — clean data, build dimensions & facts, enable V-Order.

In [ ]:
# Same Delta files are queryable by Spark AND the read-only SQL analytics endpoint — zero copy.
display(spark.sql(f"SELECT * FROM {BRONZE_SCHEMA}.sales LIMIT 10"))